# 01 — Ingestão Bronze: Clima Atual (Current Weather)
## Módulo 1A | Camada Bronze | Seção do Exame: S2 (21%)

**O que este notebook faz:**
- Tenta chamar a API OpenWeatherMap `/weather` para **65 cidades brasileiras** (27 capitais + 38 municípios da RMSP)
- Grava os payloads JSON brutos na Delta Table `workshop_weather.bronze.raw_current_weather`
- Inclui **Plano B** documentado: se o outbound internet estiver bloqueado, usa arquivos JSON locais via `COPY INTO`
- Demonstra `COPY INTO` e Auto Loader como padrões alternativos de ingestão (**Seção 2 do exame**)

**Conceito Bronze:** Gravar o dado exatamente como veio da fonte — sem nenhuma transformação. O único enriquecimento permitido são metadados de controle (`_ingestion_timestamp`, `_batch_id`, `_status`).

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — Imports, widgets e secret carregados
- [ ] Célula 2 — Função `fetch_current_weather` testada com 1 cidade
- [ ] Célula 3 — Loop sobre 4 cidades executado sem exceção não tratada
- [ ] Célula 4 — `raw_current_weather` listada em `SHOW TABLES IN workshop_weather.bronze`
- [ ] Célula 5 — `COUNT(*)` retorna 4 (ou múltiplo de 4 após re-execuções)
- [ ] Célula 6 (Plano B) — executada se API estiver bloqueada
- [ ] Célula 7 — COPY INTO demonstrado com idempotência
- [ ] Célula 8 — Auto Loader demonstrado
- [ ] Executei 2x e count chegou a 8 (append correto)

---
## Célula 1 — Imports, Widgets e Secret

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

import requests
import json
import time
import uuid
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# ── Lista completa de cidades ─────────────────────────────────────────────
# 27 capitais brasileiras (nomes sem acentos para compatibilidade com a API)
BRAZIL_CAPITALS = [
    # Norte
    "Manaus", "Belem", "Porto Velho", "Rio Branco", "Macapa", "Boa Vista", "Palmas",
    # Nordeste
    "Sao Luis", "Teresina", "Fortaleza", "Natal", "Joao Pessoa",
    "Recife", "Maceio", "Aracaju", "Salvador",
    # Centro-Oeste
    "Brasilia", "Goiania", "Cuiaba", "Campo Grande",
    # Sudeste
    "Sao Paulo", "Rio de Janeiro", "Belo Horizonte", "Vitoria",
    # Sul
    "Curitiba", "Florianopolis", "Porto Alegre",
]

# 38 municípios da Região Metropolitana de São Paulo (excluindo a capital, já listada)
RMSP_CITIES = [
    "Aruja", "Barueri", "Biritiba-Mirim", "Caieiras", "Cajamar",
    "Carapicuiba", "Cotia", "Diadema", "Embu das Artes", "Embu-Guacu",
    "Ferraz de Vasconcelos", "Francisco Morato", "Franco da Rocha", "Guararema",
    "Guarulhos", "Itapecerica da Serra", "Itapevi", "Itaquaquecetuba", "Jandira",
    "Juquitiba", "Mairipora", "Maua", "Mogi das Cruzes", "Osasco",
    "Pirapora do Bom Jesus", "Poa", "Ribeirao Pires", "Rio Grande da Serra", "Santa Isabel", "Santana de Parnaiba", "Santo Andre",
    "Sao Bernardo do Campo", "Sao Caetano do Sul", "Sao Lourenco da Serra",
    "Suzano", "Taboao da Serra", "Vargem Grande Paulista",
]

ALL_CITIES = BRAZIL_CAPITALS + RMSP_CITIES

# ── Widgets parametrizáveis ──────────────────────────────────────────────────
# Permitem sobrescrever as cidades quando este notebook roda em um Job
dbutils.widgets.text(
    "cities",
    "all",
    "Cidades (\"all\" para 65 cidades, ou separadas por vírgula)"
)
dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")

# ── Resolve a lista de cidades ─────────────────────────────────────────────
cities_input = dbutils.widgets.get("cities")
if cities_input.strip().lower() == "all":
    CITIES = ALL_CITIES
else:
    CITIES = [c.strip() for c in cities_input.split(",") if c.strip()]

CATALOG = dbutils.widgets.get("catalog")
BRONZE_TABLE = f"{CATALOG}.bronze.raw_current_weather"
VOLUME_PATH  = f"/Volumes/{CATALOG}/bronze/landing"

# ── API Key via Secrets (nunca hardcode) ─────────────────────────────────────
try:
    API_KEY = dbutils.secrets.get(scope="openweather", key="api_key")
    print("✅ API key carregada via Databricks Secrets.")
except Exception:
    # Fallback para desenvolvimento local — NUNCA commite chaves reais
    API_KEY = dbutils.widgets.get("api_key") if "api_key" in [w.name for w in dbutils.widgets.getAll()] else ""
    print("⚠️  Secret não encontrado. Configure o scope 'openweather' conforme o notebook 00.")

print(f"Cidades monitoradas : {len(CITIES)} cidades")
print(f"  Capitais          : {len(BRAZIL_CAPITALS)}")
print(f"  RMSP              : {len(RMSP_CITIES)}")
print(f"Tabela destino      : {BRONZE_TABLE}")

---
## Célula 2 — Função de Extração da API

**Boas práticas implementadas:**
- `timeout=10` — evita que o notebook trave em conexões lentas
- `raise_for_status()` — lança exceção para qualquer status HTTP 4xx/5xx
- Retorno padronizado — erros viram linhas com `_status='error'` em vez de quebrar o pipeline
- `time.sleep(1.2)` — respeita o rate limit do plano gratuito (60 req/min) com margem para 65+ cidades

In [0]:
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_current_weather(city: str, api_key: str) -> dict:
    """
    Chama o endpoint /weather e retorna um dicionário padronizado.
    Erros de rede ou API são capturados — nunca propagados como exceção.

    Retorno:
        {
            'city'       : str,
            'raw_payload': str (JSON completo como string) ou None,
            '_status'    : 'success' | 'error',
            '_error_msg' : None | str
        }
    """
    # URL sem a API key — seguro para logar
    # Sufixo ',BR' garante que a API retorne a cidade brasileira (ex: Salvador,BR e não El Salvador)
    safe_url = f"{BASE_URL}?q={city},BR&units=metric"

    try:
        response = requests.get(
            BASE_URL,
            params={"q": f"{city},BR", "appid": api_key, "units": "metric"},
            timeout=10
        )
        response.raise_for_status()  # lança HTTPError para 4xx/5xx
        return {
            "city"        : city,
            "raw_payload" : json.dumps(response.json(), ensure_ascii=False),
            "_status"     : "success",
            "_error_msg"  : None,
            "_source_url" : safe_url,
        }
    except requests.exceptions.ConnectionError as e:
        # Captura bloqueio de outbound (Free Edition) ou falha de rede
        return {
            "city"        : city,
            "raw_payload" : None,
            "_status"     : "error",
            "_error_msg"  : f"ConnectionError: {str(e)[:200]}",
            "_source_url" : safe_url,
        }
    except requests.exceptions.HTTPError as e:
        return {
            "city"        : city,
            "raw_payload" : None,
            "_status"     : "error",
            "_error_msg"  : f"HTTPError {response.status_code}: {str(e)[:200]}",
            "_source_url" : safe_url,
        }
    except Exception as e:
        return {
            "city"        : city,
            "raw_payload" : None,
            "_status"     : "error",
            "_error_msg"  : f"Unexpected: {str(e)[:200]}",
            "_source_url" : safe_url,
        }

# ── Teste isolado com uma cidade antes de rodar o loop completo ──────────────
test = fetch_current_weather("Sao Paulo", API_KEY)
print(f"Status do teste : {test['_status']}")
if test['_status'] == 'success':
    payload = json.loads(test['raw_payload'])
    print(f"Cidade retornada: {payload.get('name')} ({payload.get('sys', {}).get('country')})")
    print(f"Temperatura     : {payload.get('main', {}).get('temp')} °C")
else:
    print(f"Erro            : {test['_error_msg']}")
    print()
    print("⚠️  Se você viu ConnectionError, a API está bloqueada no Free Edition.")
    print("   Execute o PLANO B na Célula 6 para usar arquivos JSON locais.")

# ── Teste de desambiguação: Salvador deve retornar BR, não SV (El Salvador) ──
test_sv = fetch_current_weather("Salvador", API_KEY)
if test_sv['_status'] == 'success':
    sv_payload = json.loads(test_sv['raw_payload'])
    sv_country = sv_payload.get('sys', {}).get('country')
    print(f"\nSalvador → {sv_payload.get('name')} ({sv_country})")
    if sv_country == 'BR':
        print("✅ Desambiguação correta: Salvador, Bahia, Brasil")
    else:
        print(f"❌ Ainda retornando país errado: {sv_country}")

---
## Célula 3 — Loop sobre todas as cidades + montagem do batch

In [0]:
# Identificador único do batch — une todos os registros desta execução
BATCH_ID = str(uuid.uuid4())
INGESTION_TS = datetime.now(timezone.utc)

records = []
total = len(CITIES)

for i, city in enumerate(CITIES, start=1):
    result = fetch_current_weather(city, API_KEY)

    records.append({
        "city"                  : result["city"],
        "raw_payload"           : result["raw_payload"],
        "_status"               : result["_status"],
        "_error_msg"            : result["_error_msg"],
        "_source_url"           : result["_source_url"],
        "_batch_id"             : BATCH_ID,
        "_ingestion_timestamp"  : INGESTION_TS,
    })

    status_icon = "✅" if result["_status"] == "success" else "❌"
    print(f"[{i:>3}/{total}] {status_icon} {city:<30} → {result['_status']}")

    time.sleep(1.2)  # Rate limit: plano gratuito = 60 req/min (margem para 65+ cidades)

print()
success_count = sum(1 for r in records if r["_status"] == "success")
error_count   = total - success_count
print(f"Resultado: {success_count}/{total} cidades com sucesso | batch_id: {BATCH_ID[:8]}...")
if error_count > 0:
    print(f"⚠️  {error_count} cidades retornaram erro — verifique _error_msg na Bronze.")

---
## Célula 4 — Gravar na Delta Table Bronze

**Por que `mode('append')` + `mergeSchema=True`?**
- `append` — cada execução acumula dados históricos. A Bronze nunca sobrescreve.
- `mergeSchema=True` — se a API da OpenWeatherMap adicionar campos novos, a tabela evolui automaticamente sem quebrar o pipeline.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Schema explícito garante consistência entre execuções
schema = StructType([
    StructField("city",                 StringType(),    True),
    StructField("raw_payload",          StringType(),    True),
    StructField("_status",              StringType(),    False),
    StructField("_error_msg",           StringType(),    True),
    StructField("_source_url",          StringType(),    True),
    StructField("_batch_id",            StringType(),    False),
    StructField("_ingestion_timestamp", TimestampType(), False),
])

df_bronze = spark.createDataFrame(records, schema=schema)

# Grava na Delta Table — cria automaticamente se não existir
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true") # evolução automática de schema
    .saveAsTable(BRONZE_TABLE)
)

print(f"✅ Dados gravados em: {BRONZE_TABLE}")
print(f"   Linhas neste batch: {df_bronze.count()}")
df_bronze.select("city", "_status", "_ingestion_timestamp", "_batch_id").show(truncate=False)

---
## Célula 5 — Validação da Tabela Bronze

In [0]:
%sql
-- Contagem total (deve ser múltiplo de 4 após cada execução)
SELECT
    COUNT(*)                           AS total_registros,
    COUNT(DISTINCT _batch_id)          AS total_batches,
    SUM(CASE WHEN _status='success' THEN 1 ELSE 0 END) AS sucesso,
    SUM(CASE WHEN _status='error'   THEN 1 ELSE 0 END) AS erros,
    MIN(_ingestion_timestamp)          AS primeiro_batch,
    MAX(_ingestion_timestamp)          AS ultimo_batch
FROM workshop_weather.bronze.raw_current_weather;

In [0]:
%sql
-- Amostra de 1 payload bruto — confirmar que o JSON está completo
SELECT city, _status, LEFT(raw_payload, 300) AS payload_preview
FROM workshop_weather.bronze.raw_current_weather
WHERE _status = 'success'
--LIMIT 2;

---
## Célula 6 — ⚠️ PLANO B: Se a API estiver bloqueada (outbound restrito)

Se as células acima retornaram `ConnectionError`, o outbound internet da Free Edition está bloqueando `openweathermap.org`.

**Solução:** Gere os arquivos JSON **no seu computador local** e faça upload para o Volume.

### Script Python para rodar localmente (fora do Databricks):
```python
# Salve como: fetch_weather_local.py
# Execute com: python fetch_weather_local.py

import requests, json, os, time

API_KEY = "SUA_API_KEY_AQUI"  # coloque sua chave aqui — só no script local!
CITIES  = ["Sao Paulo", "Rio de Janeiro", "Curitiba", "Manaus"]
os.makedirs("weather_data/current", exist_ok=True)

for city in CITIES:
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": API_KEY, "units": "metric"},
        timeout=10
    )
    fname = f"weather_data/current/{city.replace(' ','_').lower()}.json"
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(r.json(), f, ensure_ascii=False)
    print(f"✓ {city} → {fname}")
    time.sleep(1)
```

### Upload para o Volume:
1. No Databricks: **Catalog > workshop_weather > bronze > landing > (kebab menu) > Upload to this volume**
2. Crie a pasta `sample/current/` e faça upload dos 4 arquivos `.json`
3. Execute a célula abaixo para confirmar o upload

In [0]:
# Verifica se os arquivos do Plano B estão no Volume
# import os

# sample_path = f"{VOLUME_PATH}/sample/current"

# try:
#     files = dbutils.fs.ls(sample_path)
#     print(f"✅ Arquivos encontrados em {sample_path}:")
#     for f in files:
#         print(f"   {f.name} ({f.size:,} bytes)")
# except Exception:
#     print(f"❌ Nenhum arquivo encontrado em: {sample_path}")
#     print("   Siga as instruções da Célula 6 para fazer upload dos JSONs.")

---
## Célula 7 — COPY INTO (Seção 2 do exame)

**Conceito — COPY INTO:**  
- Carrega arquivos de object storage (Volume, ADLS, S3) para uma Delta Table
- **Idempotente por design** — rastreia quais arquivos já foram carregados internamente
- Re-executar **nunca duplica** dados
- Ideal para: cargas batch periódicas de arquivos novos

**Quando usar vs alternativas:**

| Método | Quando usar |
|---|---|
| `COPY INTO` | Arquivos em object storage, carga idempotente simples |
| Auto Loader | Centenas de milhares de arquivos, schema evolution automática |
| API/JDBC | Fontes externas em tempo real ou sistemas transacionais |

In [0]:
# COPY INTO lê os JSONs do Volume e carrega em uma tabela auxiliar
# Execute somente se o Plano B foi usado (arquivos no Volume)

# copy_into_sql = f"""
# COPY INTO {CATALOG}.bronze.raw_current_weather
# FROM (
#     SELECT
#         COALESCE(name, '')                AS city,
#         to_json(struct(*))                AS raw_payload,
#         'success'                         AS _status,
#         CAST(NULL AS STRING)              AS _error_msg,
#         '{VOLUME_PATH}/sample/current'    AS _source_url,
#         '{BATCH_ID}'                      AS _batch_id,
#         current_timestamp()               AS _ingestion_timestamp
#     FROM '{VOLUME_PATH}/sample/current'
# )
# FILEFORMAT = JSON
# FORMAT_OPTIONS ('inferSchema' = 'true', 'multiLine' = 'true')
# COPY_OPTIONS ('mergeSchema' = 'true');
# """

# print("Executando COPY INTO...")
# try:
#     spark.sql(copy_into_sql)
#     print("✅ COPY INTO executado.")
#     print("   Re-execute esta célula — o count NÃO deve aumentar (idempotência).")
# except Exception as e:
#     print(f"ℹ️  {e}")
#     print("   Se os arquivos ainda não estão no Volume, ignore este erro.")

---
## Célula 8 — Auto Loader (Seção 2 do exame)

**Conceito — Auto Loader (`cloudFiles`):**
- Lê arquivos novos de um diretório de forma incremental via **Structured Streaming**
- Mantém um **checkpoint** — na próxima execução processa apenas arquivos novos
- Suporta **schema inference** e **schema evolution** automáticos
- Escala para bilhões de arquivos (diferencial vs `COPY INTO`)

> Esta célula usa `trigger(availableNow=True)` — processa os arquivos disponíveis e termina automaticamente.

In [0]:
# Auto Loader — lê JSONs do Volume de forma incremental com checkpoint
# Somente disponível se houver arquivos em /sample/current

# CHECKPOINT_PATH = f"{VOLUME_PATH}/checkpoints/autoloader_current"
# AUTOLOADER_TARGET = f"{CATALOG}.bronze.raw_autoloader_demo"

# try:
#     stream = (
#         spark.readStream
#         .format("cloudFiles")                            # Auto Loader
#         .option("cloudFiles.format", "json")             # formato dos arquivos
#         .option("cloudFiles.inferColumnTypes", "true")   # inferência de tipos
#         .option("multiLine", "true")                     # JSON multi-linha (pretty-printed)
#         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/_schema")  # persiste o schema
#         .load(f"{VOLUME_PATH}/sample/current")           # source: Volume
#     )

#     query = (
#         stream.writeStream
#         .format("delta")
#         .outputMode("append")
#         .option("checkpointLocation", CHECKPOINT_PATH)  # estado incremental
#         .trigger(availableNow=True)                     # processa tudo disponível e para
#         .toTable(AUTOLOADER_TARGET)
#     )

#     query.awaitTermination()
#     print(f"✅ Auto Loader concluído → tabela: {AUTOLOADER_TARGET}")
#     print("   Re-execute: novos arquivos serão processados; arquivos já lidos são ignorados.")

#     spark.table(AUTOLOADER_TARGET).count()
#     print(f"   Registros na tabela: {spark.table(AUTOLOADER_TARGET).count()}")

# except Exception as e:
#     print(f"ℹ️  Auto Loader não executado: {e}")
#     print("   Faça upload de arquivos JSON para o Volume e re-execute.")

---
## ✅ Checklist Final — Módulo 1A concluído?

- [ ] `raw_current_weather` existe em `workshop_weather.bronze`
- [ ] `COUNT(*)` retorna múltiplo de 4 (uma linha por cidade por execução)
- [ ] `_status = 'error'` não quebrou o pipeline — gerou linhas de auditoria
- [ ] `COPY INTO` demonstrado — re-executar **não** duplica dados
- [ ] Auto Loader demonstrado — checkpoint salvo no Volume
- [ ] Você consegue explicar quando usar cada um dos 3 métodos de ingestão

**Próximo: `02_ingest_bronze_forecast.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*